# Task 2: Sparse Evolutionary Training (SET)

In the second task, students will implement Sparse Evolutionary Training (SET), which allows the sparse connectivity pattern to evolve during training. At regular intervals during training: 
- Remove a fraction of the smallest magnitude weights
- Add the same number of new connections at random locations

This procedure maintains a constant sparsity level while enabling the network structure to adapt over time. Students should compare the performance of SET with the static sparse baseline. 

## 0. Setup and Data Loading

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import joblib

In [ ]:
# If running in Colab, uncomment these lines after uploading or mounting Drive.
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = "."  # Change to your Drive folder on Colab if needed
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

from models import ResNetCIFAR

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_pin_memory = torch.cuda.is_available()
num_workers = min(2, os.cpu_count() or 1)

print(f"Using device: {device}")
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

trainset = torchvision.datasets.CIFAR10(
    root=os.path.join(PROJECT_DIR, "data"),
    train=True,
    download=True,
    transform=transform,
)
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
)

testset = torchvision.datasets.CIFAR10(
    root=os.path.join(PROJECT_DIR, "data"),
    train=False,
    download=True,
    transform=transform,
)
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
)

classes = trainset.classes
print(f"Loaded CIFAR-10 with {len(classes)} classes.")

## 1. SET Configuration

In [ ]:
sparsity_levels = [0.90, 0.95, 0.98]
optimizers = ["sgd", "adam"]
prune_fraction = 0.30
rewire_interval = 1
epochs = 15

criterion = nn.CrossEntropyLoss()
output_dir = os.path.join(PROJECT_DIR, "results")
os.makedirs(output_dir, exist_ok=True)

print(f"Sparsity levels: {sparsity_levels}")
print(f"Optimizers: {optimizers}")
print(f"Prune fraction: {prune_fraction:.2f}")
print(f"Rewire interval: every {rewire_interval} epoch(s)")
print(f"Output directory: {output_dir}")

In [ ]:
def initialize_masks(model, sparsity):
    """Initialize binary masks for weight tensors based on target sparsity."""
    masks = {}
    for name, param in model.named_parameters():
        if "weight" in name and param.ndim > 1:
            mask = (torch.rand_like(param) > sparsity).float()
            masks[name] = mask
    return masks


def apply_masks(model, masks):
    """Zero out pruned weights by applying masks in-place."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in masks:
                param.mul_(masks[name])


def apply_masks_to_grads(model, masks):
    """Keep pruned connections frozen by masking their gradients."""
    for name, param in model.named_parameters():
        if name in masks and param.grad is not None:
            param.grad.mul_(masks[name])


def prune_and_regrow(model, masks, prune_fraction):
    """Magnitude prune active weights, then randomly regrow the same count."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name not in masks:
                continue

            weights = param
            mask = masks[name]

            active_positions = mask.nonzero(as_tuple=False)
            if active_positions.numel() == 0:
                continue

            active_weights = weights[mask.bool()]
            n_prune = int(active_weights.numel() * prune_fraction)
            if n_prune < 1:
                continue

            _, prune_indices = torch.topk(
                active_weights.abs(), k=n_prune, largest=False
            )
            prune_positions = active_positions[prune_indices]
            prune_index = tuple(
                prune_positions[:, dim] for dim in range(prune_positions.size(1))
            )

            mask[prune_index] = 0.0
            weights[prune_index] = 0.0

            zero_positions = (mask == 0).nonzero(as_tuple=False)
            if zero_positions.numel() == 0:
                continue

            n_grow = min(n_prune, zero_positions.size(0))
            grow_choices = torch.randperm(
                zero_positions.size(0), device=zero_positions.device
            )[:n_grow]
            grow_positions = zero_positions[grow_choices]
            grow_index = tuple(
                grow_positions[:, dim] for dim in range(grow_positions.size(1))
            )

            mask[grow_index] = 1.0
            std = weights.std()
            if std == 0:
                std = 0.01
            weights[grow_index] = torch.randn(n_grow, device=weights.device) * std


def build_sparse_model(sparsity):
    model = ResNetCIFAR().to(device)
    masks = initialize_masks(model, sparsity)
    apply_masks(model, masks)
    return model, masks

## 2. Train and Save Results

In [ ]:
def build_optimizer(model, optimizer_type):
    if optimizer_type == "sgd":
        return optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    if optimizer_type == "adam":
        return optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    raise ValueError(f"Unsupported optimizer: {optimizer_type}")


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total_examples += labels.size(0)
            total_correct += predicted.eq(labels).sum().item()

    average_loss = total_loss / len(loader)
    accuracy = 100.0 * total_correct / total_examples
    return average_loss, accuracy


all_results = {}

for opt_name in optimizers:
    all_results[opt_name] = {}
    for target_sparsity in sparsity_levels:
        print(
            f"\n===== SET Run | Sparsity={target_sparsity:.2f} | Optimizer={opt_name} ====="
        )

        model, masks = build_sparse_model(target_sparsity)
        optimizer = build_optimizer(model, opt_name)
        history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            for inputs, labels in trainloader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                apply_masks_to_grads(model, masks)
                optimizer.step()
                apply_masks(model, masks)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

            if (epoch + 1) % rewire_interval == 0:
                prune_and_regrow(model, masks, prune_fraction)
                apply_masks(model, masks)

            total_active = sum(mask.sum().item() for mask in masks.values())
            total_possible = sum(mask.numel() for mask in masks.values())
            current_sparsity = 1 - total_active / total_possible

            train_loss = running_loss / len(trainloader)
            train_acc = 100.0 * correct / total
            test_loss, test_acc = evaluate(model, testloader, criterion)

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["test_loss"].append(test_loss)
            history["test_acc"].append(test_acc)

            print(
                f"Epoch [{epoch + 1}/{epochs}] | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.2f}% | "
                f"Test Loss: {test_loss:.4f} | "
                f"Test Acc: {test_acc:.2f}% | "
                f"Current Sparsity: {current_sparsity:.4f}"
            )

        experiment_results = {
            "sparsity": target_sparsity,
            "optimizer": opt_name,
            "method": "SET",
            "prune_fraction": prune_fraction,
            "rewire_interval": rewire_interval,
            "epochs": epochs,
            "train_loss": history["train_loss"],
            "train_acc": history["train_acc"],
            "test_loss": history["test_loss"],
            "test_acc": history["test_acc"],
            "masks": masks,
            "model_state": model.state_dict(),
        }

        results_path = os.path.join(
            output_dir,
            f"task_2_{opt_name}_{int(target_sparsity * 100)}_sparsity.joblib",
        )
        joblib.dump(experiment_results, results_path)
        print(f"Saved results to {results_path}")

        all_results[opt_name][target_sparsity] = experiment_results

print("\nFinished all sparsity and optimizer runs.")